<a href="https://colab.research.google.com/github/brasilemnumeros/dados/blob/main/indicadores/saude/MortalidadeInfantil.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mortalidade infantil — taxa por mil nascidos vivos

Este notebook é o motor de coleta e publicação da série `mortalidade-infantil-anual`.

- **Fonte:** Ministério da Saúde/Secretaria de Vigilância em Saúde e Ambiente (MS/SVSA) —
  microdados do Sistema de Informações sobre Mortalidade (SIM) e do Sistema de Informações
  sobre Nascidos Vivos (SINASC), distribuídos pelo DATASUS via FTP (`ftp.datasus.gov.br`).
  - Numerador: óbitos de menores de 1 ano de idade — arquivo nacional `DOINF{AA}.dbc` em
    `dissemin/publicos/SIM/CID10/DOFET/` (um por ano).
  - Denominador: nascidos vivos — arquivo nacional `DNBR{AAAA}.dbc` em
    `dissemin/publicos/SINASC/1996_/Dados/DNRES/` (um por ano).
- **Definição:** número de óbitos de menores de 1 ano por mil nascidos vivos, no
  ano-calendário, por residência. É o cálculo **direto** a partir das contagens oficiais do
  SIM e do SINASC, **sem correção de sub-registro**.
- **Fórmula:** `taxa = 1000 × óbitos_de_menores_de_1_ano ÷ nascidos_vivos`, por ano.
- **Unidade:** por mil nascidos vivos (`unidade.tipo: numero`, `simbolo: por mil`,
  `casas: 1`); `eixo: por-mil`, `medida: nivel`, `granularidade: anual`.
- **Período publicado:** `1996-01-01` até o último ano com SIM e SINASC **finais**.
- **Arquivo gerado:** `indicadores/saude/mortalidade-infantil-anual.json`.

## O que este número é — e o que ele não é

- **É o cálculo direto, sem correção de sub-registro.** Por isso fica sistematicamente
  abaixo da taxa indireta do IBGE (série CD100, 2000–2016), que corrige o sub-registro a
  partir do Censo: em 2000 a diferença é de ~7,7 por mil e em 2016 de ~0,6 por mil. A taxa
  direta é a que o MS e os boletins estaduais publicam como "taxa de mortalidade infantil"
  (12,62 em 2023, 12,56 em 2024).
- **Não é a taxa neonatal nem a de menores de 5 anos** — só o primeiro ano de vida.
- **Os anos recentes são revisados.** O SIM e o SINASC consolidam as declarações com atraso:
  um ano só entra na série quando os dois sistemas o publicam como final. Reexecutar este
  notebook muda valores apenas se o DATASUS revisar a base; aí é revisão de fonte,
  registrada em `docs/metodologia.md`.

## Por que microdados e não o TabNet

A taxa pronta do TabNet do SIM/SINASC (`sim/cnv/inf10uf.def`) não está scriptável hoje: o
servidor devolve "Tabela de conversão não encontrada" ao tabular. Por isso o notebook
recalcula a taxa a partir dos **microdados** (as contagens oficiais): baixa os arquivos
`DOINF` (SIM) e `DNBR` (SINASC) do FTP do DATASUS, conta os registros de cada um e aplica a
fórmula acima. A fórmula fica visível e auditável.

In [1]:
from pathlib import Path
import ftplib
import json
import math
import os
import re
import struct
import tempfile
import time
from decimal import Decimal, ROUND_HALF_UP

from pyreaddbc.readdbc import dbc2dbf

RAIZ = Path(os.environ.get("DADOS_RAIZ", Path.cwd())).resolve()
if os.environ.get("DADOS_RAIZ"):
    PASTA = RAIZ / "indicadores" / "saude"
    if not PASTA.is_dir():
        raise SystemExit(f"família não encontrada em {PASTA}")
else:
    PASTA = RAIZ
DESTINO = Path(os.environ.get("SAUDE_SAIDA", PASTA))


def relativo(caminho):
    """Caminho exibido no notebook: relativo à raiz, nunca o absoluto da máquina."""
    try:
        return caminho.relative_to(RAIZ)
    except ValueError:
        return caminho.name


FTP_HOST = "ftp.datasus.gov.br"
# Diretórios dos microdados FINAIS (o ano só entra quando SIM e SINASC o publicam como final).
DIR_SIM = "/dissemin/publicos/SIM/CID10/DOFET"
DIR_SINASC = "/dissemin/publicos/SINASC/1996_/Dados/DNRES"
INICIO = 1996

In [2]:
def listar(caminho):
    """Lista um diretório do FTP DATASUS, com reconexão e espera crescente."""
    for tentativa in range(1, 6):
        try:
            ftp = ftplib.FTP(FTP_HOST, timeout=60)
            ftp.login()
            itens = []
            ftp.retrlines("NLST " + caminho, itens.append)
            ftp.quit()
            return [Path(item).name for item in itens]
        except Exception:
            if tentativa == 5:
                raise
            time.sleep(2 ** tentativa)


def baixar(caminho, destino):
    """Baixa um arquivo do FTP DATASUS para ``destino``, com reconexão e retry."""
    for tentativa in range(1, 6):
        try:
            ftp = ftplib.FTP(FTP_HOST, timeout=60)
            ftp.login()
            with open(destino, "wb") as saida:
                ftp.retrbinary("RETR " + caminho, saida.write, blocksize=1024 * 1024)
            ftp.quit()
            return
        except Exception:
            if tentativa == 5:
                raise
            time.sleep(2 ** tentativa)


def contar_registros(caminho_dbc):
    """Devolve o número de registros ativos de um .dbc do DATASUS.

    Descomprime o .dbc (dBase comprimido) com o ``pyreaddbc`` e lê o cabeçalho dBase: o
    total de registros está nos bytes 4–7 (little-endian); registros marcados como
    apagados (flag 0x2A no primeiro byte) são descontados.
    """
    fd, caminho_dbf = tempfile.mkstemp(suffix=".dbf")
    os.close(fd)
    try:
        dbc2dbf(str(caminho_dbc), caminho_dbf)
        with open(caminho_dbf, "rb") as arquivo:
            cabecalho = arquivo.read(32)
        total = struct.unpack("<I", cabecalho[4:8])[0]
        comprimento = struct.unpack("<H", cabecalho[10:12])[0]
        inicio_dados = struct.unpack("<H", cabecalho[8:10])[0]
        with open(caminho_dbf, "rb") as arquivo:
            arquivo.seek(inicio_dados)
            dados = arquivo.read(total * comprimento)
        apagados = dados[0::comprimento].count(b"\x2a")
        return total - apagados
    finally:
        os.remove(caminho_dbf)


def ano_do_doinf(nome):
    m = re.fullmatch(r"DOINF(\d{2})\.dbc", nome, re.IGNORECASE)
    if not m:
        return None
    aa = int(m.group(1))
    # 96–99 → 1996–1999; 00–25 → 2000–2025 (a janela da base, não um limite rígido).
    return 2000 + aa if aa <= 25 else 1900 + aa


def ano_do_dnbr(nome):
    m = re.fullmatch(r"DNBR(\d{4})\.dbc", nome, re.IGNORECASE)
    return int(m.group(1)) if m else None


obitos_por_ano = {}
for nome in listar(DIR_SIM):
    ano = ano_do_doinf(nome)
    if ano is not None and ano >= INICIO:
        obitos_por_ano[ano] = f"{DIR_SIM}/{nome}"

nascidos_por_ano = {}
for nome in listar(DIR_SINASC):
    ano = ano_do_dnbr(nome)
    if ano is not None and ano >= INICIO:
        nascidos_por_ano[ano] = f"{DIR_SINASC}/{nome}"

# Um ano só entra quando o SIM e o SINASC o publicam como final; o ano corrente (ainda
# preliminar) e anos com só um dos lados final ficam de fora.
anos = sorted(set(obitos_por_ano) & set(nascidos_por_ano))
print(f"anos com SIM e SINASC finais: {anos[0]}–{anos[-1]} ({len(anos)} anos)")

anos com SIM e SINASC finais: 1996–2024 (29 anos)


In [3]:
obitos = {}
nascidos = {}
for ano in anos:
    with tempfile.TemporaryDirectory() as pasta:
        doinf = Path(pasta) / "doinf.dbc"
        dnbr = Path(pasta) / "dnbr.dbc"
        baixar(obitos_por_ano[ano], doinf)
        baixar(nascidos_por_ano[ano], dnbr)
        obitos[ano] = contar_registros(doinf)
        nascidos[ano] = contar_registros(dnbr)
    print(f"{ano}: {obitos[ano]:,} óbitos <1 ano, {nascidos[ano]:,} nascidos vivos")


def arredondar(valor):
    """Arredonda para 2 casas com meio para cima (convenção comercial)."""
    return float(Decimal(repr(valor)).quantize(Decimal("0.01"), rounding=ROUND_HALF_UP))


# Fórmula: taxa = 1000 × óbitos de menores de 1 ano ÷ nascidos vivos (por residência).
taxa = {ano: arredondar(1000.0 * obitos[ano] / nascidos[ano]) for ano in anos}

print()
print(f"{'ano':>4} {'óbitos':>9} {'nascidos':>11} {'taxa':>6}")
for ano in anos:
    print(f"{ano:>4} {obitos[ano]:>9,} {nascidos[ano]:>11,} {taxa[ano]:>6.2f}")

1996: 75,024 óbitos <1 ano, 2,945,425 nascidos vivos


1997: 71,400 óbitos <1 ano, 3,026,658 nascidos vivos


1998: 71,690 óbitos <1 ano, 3,148,037 nascidos vivos


1999: 69,345 óbitos <1 ano, 3,256,433 nascidos vivos


2000: 68,199 óbitos <1 ano, 3,206,761 nascidos vivos


2001: 61,943 óbitos <1 ano, 3,115,474 nascidos vivos


... (19 outputs left out)

2021: 31,856 óbitos <1 ano, 2,677,101 nascidos vivos


2022: 32,257 óbitos <1 ano, 2,561,922 nascidos vivos


2023: 32,017 óbitos <1 ano, 2,537,576 nascidos vivos


2024: 30,020 óbitos <1 ano, 2,389,325 nascidos vivos

 ano    óbitos    nascidos   taxa
1996    75,024   2,945,425  25.47
1997    71,400   3,026,658  23.59
1998    71,690   3,148,037  22.77
1999    69,345   3,256,433  21.29
2000    68,199   3,206,761  21.27
2001    61,943   3,115,474  19.88
2002    58,916   3,059,402  19.26
2003    57,540   3,038,251  18.94
2004    54,183   3,026,548  17.90
2005    51,544   3,035,096  16.98
2006    48,332   2,944,928  16.41
2007    45,370   2,891,328  15.69
2008    44,100   2,934,828  15.03
2009    42,661   2,881,581  14.80
2010    39,874   2,861,868  13.93
2011    39,716   2,913,160  13.63
2012    39,123   2,905,789  13.46
2013    38,966   2,904,027  13.42
2014    38,432   2,979,259  12.90
2015    37,501   3,017,668  12.43
2016    36,350   2,857,800  12.72
2017    36,223   2,923,535  12.39
2018    35,864   2,944,932  12.18
2019    35,293   2,849,146  12.39
2020    31,439   2,730,145  11.52
2021    31,856   2,677,101  11.90
2022    32,257   2,561,922  

In [4]:
def pontos(serie):
    return [
        {"data": f"{ano}-01-01", "valor": valor}
        for ano, valor in sorted(serie.items())
    ]


FONTE = (
    "MS/SVSA — SIM (óbitos de menores de 1 ano) e SINASC (nascidos vivos), microdados do "
    "DATASUS; taxa por mil nascidos vivos calculada pelo site: 1000 × óbitos ÷ nascidos vivos"
)
URL_FONTE = "https://datasus.saude.gov.br/transferencia-de-arquivos/"

serie = {
    "versao": 1,
    "id": "mortalidade-infantil-anual",
    "unidade": {"tipo": "numero", "simbolo": "por mil", "casas": 1},
    "eixo": "por-mil",
    "medida": "nivel",
    "granularidade": "anual",
    "fonte": FONTE,
    "urlFonte": URL_FONTE,
    "dados": pontos(taxa),
}

DESTINO.mkdir(parents=True, exist_ok=True)
arquivo = DESTINO / "mortalidade-infantil-anual.json"
with arquivo.open("w", encoding="utf-8") as saida:
    json.dump(serie, saida, ensure_ascii=False, indent=2)
    saida.write("\n")
print(f"{len(serie['dados'])} pontos escritos em {relativo(arquivo)}")

29 pontos escritos em .superpowers/sdd/2026-11-04-quatro-indicadores-sociais/ensaio-t4/mortalidade-infantil-anual.json


In [5]:
CHAVES_RAIZ = {
    "versao", "id", "unidade", "eixo", "medida", "granularidade", "fonte", "urlFonte", "dados",
}

assert set(serie) == CHAVES_RAIZ, sorted(set(serie) ^ CHAVES_RAIZ)
assert serie["versao"] == 1
assert serie["id"] == "mortalidade-infantil-anual"
assert serie["unidade"] == {"tipo": "numero", "simbolo": "por mil", "casas": 1}
assert serie["eixo"] == "por-mil"
assert serie["medida"] == "nivel"
assert serie["granularidade"] == "anual"
assert isinstance(serie["fonte"], str) and serie["fonte"]
assert serie["urlFonte"].startswith("https://")

dados = serie["dados"]
assert dados, "série vazia"
datas = [ponto["data"] for ponto in dados]
anos_lista = [int(data[:4]) for data in datas]

assert datas[0] == "1996-01-01", datas[0]
assert all(data.endswith("-01-01") for data in datas), "dia diferente de 01"
assert all(a < b for a, b in zip(datas, datas[1:])), "datas fora de ordem"
assert anos_lista == list(range(anos_lista[0], anos_lista[-1] + 1)), "ano faltando ou repetido"

for ponto in dados:
    assert set(ponto) == {"data", "valor"}, sorted(ponto)
    assert not isinstance(ponto["valor"], bool), ponto
    assert isinstance(ponto["valor"], (int, float)), ponto
    assert math.isfinite(ponto["valor"]), ponto

# Fórmula: cada ponto é 1000 × óbitos ÷ nascidos do mesmo ano.
for ponto in dados:
    ano = int(ponto["data"][:4])
    assert ponto["valor"] == arredondar(1000.0 * obitos[ano] / nascidos[ano]), ponto

# Confere a gravação em disco.
with (DESTINO / "mortalidade-infantil-anual.json").open(encoding="utf-8") as arquivo:
    assert json.load(arquivo) == serie, "arquivo gravado difere do objeto"


def valor(ano):
    return {int(p["data"][:4]): p["valor"] for p in dados}[ano]


# Cobertura: primeiro e último ponto.
assert valor(1996) == 25.47
assert dados[-1]["data"] == "2024-01-01", dados[-1]["data"]
assert len(dados) == 29, len(dados)

# Números citados na metodologia e na evidência (taxa direta SIM/Sinasc).
assert valor(2000) == 21.27
assert valor(2016) == 12.72
assert valor(2020) == 11.52
assert valor(2023) == 12.62
assert valor(2024) == 12.56

print(
    f"ok: {len(dados)} pontos ({datas[0]} a {datas[-1]}); fórmula, dia 01, ordem, "
    "cobertura e contrato conferidos."
)

ok: 29 pontos (1996-01-01 a 2024-01-01); fórmula, dia 01, ordem, cobertura e contrato conferidos.
